In [ ]:
import os
import shutil
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import urllib.parse

load_dotenv()
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

safe_password = urllib.parse.quote_plus(DB_PASSWORD)
db_url = f"mysql+pymysql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)


In [ ]:
# 두 테이블에서 데이터를 각각 로드
query_vision = "SELECT LOTID, FILEPATH FROM ai_vision_davalue"
query_proc = "SELECT * FROM ai_proc_prevalue"

df_vision = pd.read_sql(query_vision, engine)
df_proc = pd.read_sql(query_proc, engine)

# LOTID를 기준으로 교집합(Inner Join) 병합
df_merged = pd.merge(df_vision, df_proc, on='LOTID', how='inner')
print(f"▶ 병합된 전체 유효 데이터 수: {len(df_merged)}건")

In [ ]:
# random_state를 고정하여 항상 동일한 2,000건이 추출되도록 설정
df_sampled = df_merged.sample(n=2000, random_state=42).reset_index(drop=True)

# 데이터 조합 결과 확인
print(f"▶ 샘플링 완료: {len(df_sampled)}건")
display(df_sampled.head())

In [ ]:
# 컬럼명과 데이터 타입 출력
print("=== 데이터프레임 구조 확인 ===")
print(df_sampled.dtypes)

# 누락된 데이터(결측치)가 있는지 함께 확인
print("\n=== 데이터 정보 요약 ===")
df_sampled.info()

In [ ]:
TARGET_DB_USER = "****"
TARGET_DB_PASSWORD = "****"
TARGET_DB_HOST = "****"
TARGET_DB_PORT = "****"
TARGET_DB_NAME = "****"

new_safe_password = urllib.parse.quote_plus(TARGET_DB_PASSWORD)
target_db_url = f"mysql+pymysql://{TARGET_DB_USER}:{new_safe_password}@{TARGET_DB_HOST}:{TARGET_DB_PORT}/{TARGET_DB_NAME}"
target_engine = create_engine(target_db_url)

table_name = 'wm_unstructed_datasets'

print("새로운 DB에 테이블 생성 및 데이터 적재 중...")
df_sampled.to_sql(
    name=table_name,
    con=target_engine,
    if_exists='replace',
    index=False
)

# 3. LOTID 컬럼을 PRIMARY KEY로 변경 및 공백 제거(필수 전처리)
# 기존에 테이블이 자동 생성되면 기본키가 없는 상태이므로, DDL 명령으로 PK를 주입합니다.
print("LOTID를 Primary Key로 설정 중...")
with target_engine.begin() as connection:
    # LOTID 컬럼의 데이터 타입을 VARCHAR(50)로 명확히 지정하며 고유키 조건 부여
    connection.execute(text(f"ALTER TABLE {table_name} MODIFY LOTID VARCHAR(50);"))
    connection.execute(text(f"ALTER TABLE {table_name} ADD PRIMARY KEY (LOTID);"))

print(f"▶ [완료] 새로운 DB({TARGET_DB_NAME}) 내 {table_name} 테이블 생성 및 PK(LOTID) 설정 완료.")

In [ ]:
target_dir = "C:/Users"
os.makedirs(target_dir, exist_ok=True)

new_paths = []
success_count = 0
fail_count = 0

print(f"[{target_dir}] 디렉터리로 이미지 복사를 시작합니다...")

# 2. 데이터프레임을 순회하며 파일 복사 (tqdm 생략)
for idx, row in df_sampled.iterrows():
    old_path = row['FILEPATH']
    
    # 진행 상황 단순 텍스트 출력 (500건 단위)
    if (idx + 1) % 500 == 0:
        print(f"... 진행 중: {idx + 1} / {len(df_sampled)} 건 처리 완료")
        
    # 경로값이 존재하고 실제 원본 파일이 있는 경우에만 복사 진행
    if pd.notna(old_path) and os.path.exists(old_path):
        filename = os.path.basename(old_path)
        new_path = os.path.join(target_dir, filename).replace("\\", "/")
        
        try:
            # 원본 파일의 메타데이터를 유지하며 복사
            shutil.copy2(old_path, new_path)
            new_paths.append(new_path)
            success_count += 1
        except Exception as e:
            # 에러 로그가 너무 많이 뜨면 불편하므로 실패 카운트만 올립니다
            new_paths.append(None)
            fail_count += 1
    else:
        new_paths.append(None)
        fail_count += 1

# 3. 복사된 새로운 경로를 데이터프레임의 기존 컬럼에 덮어쓰기
df_sampled['FILE_PATH'] = new_paths

print(f"\n▶ 파일 복사 완료: 성공 {success_count}건 | 누락/실패 {fail_count}건")

In [ ]:
df_valid = df_sampled.dropna(subset=['FILEPATH']).copy()

import os
from sqlalchemy import text

# 1. 새로운 기준 디렉터리 지정
target_dir = "C:/Users"

# 2. 기존 FILEPATH에서 파일명만 추출하여 새로운 디렉터리와 결합
df_valid['FILEPATH'] = df_valid['FILEPATH'].apply(
    lambda x: os.path.join(target_dir, os.path.basename(str(x))).replace("\\", "/")
)

# 3. 변환된 경로 확인 (상위 5개)
print("▶ 변환된 FILEPATH 확인:")
display(df_valid[['LOTID', 'FILEPATH']].head())

# 4. DB에 덮어쓰기
df_valid.to_sql(
    name='wm_unstructed_datasets',
    con=target_engine,
    if_exists='replace',
    index=False
)

# 5. PK 재설정
print("▶ 새로운 경로 DB 업데이트 후 LOTID Primary Key 재설정 중...")
with target_engine.begin() as connection:
    connection.execute(text("ALTER TABLE wm_unstructed_datasets MODIFY LOTID VARCHAR(50);"))
    connection.execute(text("ALTER TABLE wm_unstructed_datasets ADD PRIMARY KEY (LOTID);"))

print("▶ 최종 완료: 파일 경로 업데이트 및 DB 동기화 성공!")

In [ ]:
# 1. 이미 파일들이 존재하는 C드라이브 타겟 디렉터리
target_dir = "C:/Users"

new_filenames = []
new_filepaths = []
success_count = 0

print(f"▶ [{target_dir}] 내의 파일명 변경(TEST_1 ~ TEST_2000)을 시작합니다...")

# 데이터프레임 인덱스 초기화 (1번부터 깔끔하게 부여하기 위함)
df_valid = df_valid.reset_index(drop=True)

# 2. 데이터프레임 순회하며 C드라이브 내의 실제 파일 이름 변경
for idx, row in df_valid.iterrows():
    # 기존 컬럼이 FILE_PATH인지 FILEPATH인지 방어적으로 가져오기
    old_db_path = row.get('FILE_PATH') if 'FILE_PATH' in row else row.get('FILEPATH')
    
    # DB에 적혀있던 원본 파일명만 추출 (예: 20251225...9670.jpg)
    original_filename = os.path.basename(str(old_db_path))
    
    # 현재 C드라이브에 있는 실제 원본 파일의 전체 경로
    current_physical_path = os.path.join(target_dir, original_filename).replace("\\", "/")
    
    # 확장자 추출 및 새로운 TEST_N 파일명 생성
    _, ext = os.path.splitext(original_filename)
    if not ext: ext = '.jpg'
    
    new_filename = f"TEST_{idx + 1}{ext}"
    new_physical_path = os.path.join(target_dir, new_filename).replace("\\", "/")
    
    # 3. 물리적 파일 이름 변경 (같은 C드라이브 내부라 에러 없이 즉각 처리됨)
    if os.path.exists(current_physical_path):
        try:
            os.rename(current_physical_path, new_physical_path)
            success_count += 1
        except Exception as e:
            print(f"[오류] 파일 변경 실패 ({original_filename}): {e}")
    else:
        print(f"[경고] C드라이브 폴더에서 파일을 찾을 수 없습니다: {original_filename}")
        
    new_filenames.append(new_filename)
    new_filepaths.append(new_physical_path)

# 4. 데이터프레임 컬럼 정리 및 갱신
df_valid['FILENAME'] = new_filenames
df_valid['FILEPATH'] = new_filepaths
if 'FILE_PATH' in df_valid.columns:
    df_valid = df_valid.drop(columns=['FILE_PATH']) # 옛날 컬럼명 삭제

print(f"\n▶ 파일명 변경 완료: 총 {success_count}건 성공")

# 5. DB에 한 번에 업데이트 및 PK 설정
table_name = 'wm_unstructed_datasets'
print(f"▶ DB 테이블({table_name})에 덮어쓰기 및 LOTID 기본키 설정 중...")

df_valid.to_sql(
    name=table_name,
    con=target_engine,
    if_exists='replace',
    index=False
)

with target_engine.begin() as connection:
    connection.execute(text(f"ALTER TABLE {table_name} MODIFY LOTID VARCHAR(50);"))
    connection.execute(text(f"ALTER TABLE {table_name} ADD PRIMARY KEY (LOTID);"))

print("▶ [최종 완료] 파일명 변경, 경로 수정, DB 업데이트가 모두 성공적으로 끝났습니다!")
display(df_valid[['LOTID', 'FILENAME', 'FILEPATH']].head())